In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.utils import resample
import os

In [ ]:
df = pd.read_csv('C:/Users/starc/OneDrive/Documentos/Extracción de conocimiento en bases de datos/1a.-Evaluaci-n/data.csv', encoding='latin-1', low_memory=False)
df_clean = df.dropna(subset=['CustomerID'])
df_clean = df_clean[(df_clean['Quantity'] > 0) & (df_clean['UnitPrice'] > 0)]
df_clean = df_clean[~df_clean['StockCode'].str.contains('POST|BANK CHARGES', case=False, na=False)]

In [5]:
df_clean['TotalAmount'] = df_clean['Quantity'] * df_clean['UnitPrice']
invoice = df_clean.groupby('InvoiceNo').agg({
    'TotalAmount': 'sum',
    'CustomerID': 'first',
    'Country': 'first'
}).reset_index()

percentil_75 = invoice['TotalAmount'].quantile(0.75)
invoice['CONTRATO_CRITICO'] = invoice['TotalAmount'] > percentil_75

In [6]:
features = df_clean.groupby('InvoiceNo').agg({
    'Quantity': 'sum',
    'StockCode': 'nunique',
    'UnitPrice': 'mean'
}).reset_index()

features.columns = ['InvoiceNo', 'Total_Items', 'Unique_Products', 'Avg_Price']
data = invoice.merge(features, on='InvoiceNo')
data = pd.get_dummies(data, columns=['Country'], drop_first=True)

X = data.drop(['InvoiceNo', 'CustomerID', 'TotalAmount', 'CONTRATO_CRITICO'], axis=1)
y = data['CONTRATO_CRITICO']

In [7]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

In [8]:
train_data = pd.concat([X_train, y_train], axis=1)
minority = train_data[train_data['CONTRATO_CRITICO'] == True]
majority = train_data[train_data['CONTRATO_CRITICO'] == False]

if len(minority) > 0:
    minority_upsampled = resample(minority, replace=True, n_samples=len(majority), random_state=42)
    train_balanced = pd.concat([majority, minority_upsampled]).sample(frac=1, random_state=42).reset_index(drop=True)
    X_train_bal = train_balanced.drop('CONTRATO_CRITICO', axis=1)
    y_train_bal = train_balanced['CONTRATO_CRITICO']
else:
    X_train_bal, y_train_bal = X_train, y_train

In [9]:
os.makedirs('datos_mineria', exist_ok=True)
X_train_bal.to_csv('datos_mineria/X_train.csv', index=False)
y_train_bal.to_csv('datos_mineria/y_train.csv', index=False)
X_test.to_csv('datos_mineria/X_test.csv', index=False)
y_test.to_csv('datos_mineria/y_test.csv', index=False)
data.to_csv('data_con_target.csv', index=False)

In [10]:
print('Archivos exportados correctamente')
print(f'Train - Críticos: {y_train_bal.mean()*100:.1f}%')
print(f'Test - Críticos: {y_test.mean()*100:.1f}%')
print(f'X_train: {X_train_bal.shape}, y_train: {y_train_bal.shape}')
print(f'X_test: {X_test.shape}, y_test: {y_test.shape}')

Archivos exportados correctamente
Train - Críticos: 50.0%
Test - Críticos: 25.0%
X_train: (19392, 39), y_train: (19392,)
X_test: (5541, 39), y_test: (5541,)


In [11]:
print('Archivos en carpeta datos_mineria:')
for file in os.listdir('datos_mineria'):
    print(f'  - {file}')

Archivos en carpeta datos_mineria:
  - X_test.csv
  - X_train.csv
  - y_test.csv
  - y_train.csv


In [12]:
print('\nMuestra de X_train:')
print(X_train_bal.head(3))
print('\nMuestra de y_train:')
print(y_train_bal.head(3))


Muestra de X_train:
   Total_Items  Unique_Products  Avg_Price  Country_Austria  Country_Bahrain  \
0          137               15   2.576000            False            False   
1          467               33   2.688788            False            False   
2           92               34   2.928824            False            False   

   Country_Belgium  Country_Brazil  Country_Canada  Country_Channel Islands  \
0            False           False           False                    False   
1            False           False           False                    False   
2            False           False           False                    False   

   Country_Cyprus  ...  Country_RSA  Country_Saudi Arabia  Country_Singapore  \
0           False  ...        False                 False              False   
1           False  ...        False                 False              False   
2           False  ...        False                 False              False   

   Country_Spain  Co